### Comparison of the weekly air quality data from different devices

The data will first be cleaned so as to be easily transfered to tableau for vizualizations

In [1]:
#Operational packages 
import numpy as np 
import pandas as pd
import janitor

#Data viz packages 
import matplotlib.pyplot as plt
import seaborn as sns

### Data exploration

In [2]:
#Enable the display of the entire dataset whenever possible 
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

#Bring in the two datasets; clarity and quantaq
clarity_2devices_data = pd.read_csv("clarity-hourly-A93NM49L-A44MFTF3-11-09-to-11-15-2025.csv")
clarity_2devices_data_rpt = pd.read_csv("clarity-hourly-report-A93NM49L-A44MFTF3-11-09-to-11-15-2025.csv")
quantaq_959_data = pd.read_csv("quantaq-hourly-MOD-X-00959-HOUR-11-09-to-11-15-2025.csv")

#Remove empty columns 
clarity_recorded_col = clarity_2devices_data.dropna(axis='columns', how="all")
clarity_recorded_col_rpt = clarity_2devices_data_rpt.dropna(axis='columns', how="all")
quantaq_recorded_col = quantaq_959_data.dropna(axis='columns', how="all")

#Keep only the columns that are likely to be useful in data viz
clarity_col_reduced = clarity_recorded_col_rpt[['sourceId', 'outputFrequency', 'startOfPeriod', 
                                                'endOfPeriod','no2Conc1HourMean.raw',
                                                'relHumidInternal1HourMean.raw',
                                                'temperatureInternal1HourMean.raw',]]

#Rename the columns and show distinguish them by manufacture
clarity_col_renamed = clarity_col_reduced.set_axis(['sourceIdClarity', 
                                                    'outputFrequency', 
                                                    'startOfPeriodUtcClarity', 
                                                    'endOfPeriodUtcClarity','no2Conc1HourMeanUtcClarity',
                                                    'relHumidInternal1HourMeanClarity',
                                                    'temperatureInternal1HourMeanClarity'], axis=1)


clarity_col_renamed = clarity_col_renamed.clean_names(case_type='snake')

#Slect only the required columns from quantaq and clean them too
quantaq_col_reduced = quantaq_recorded_col[['period_start_utc','period_end_utc',
                                            'sn', 'n_datapoints', 'sample_rh', 
                                            'sample_temp', 'no2']]
quantaq_col_renamed = quantaq_col_reduced.set_axis(['start_of_period_utc_qa', 'end_of_period_utc_qa',                                             
                                                    'source_id_qa', 'n_datapoints_qa', 'rel_humid_qa', 
                                                    'temperature_qa', 'no2_qa'], axis=1)


### Records cleaning 
Clean and standardize the records in different fields 

In [3]:
def date_converter(data_frame):
    """
    A function that gets an air quality dataframe, looks for date columns 
    and converts them into the right the proper data type

    """

    for col in data_frame.columns:
        if 'period' in col:
            data_frame[col] = pd.to_datetime(data_frame[col])   
        data_frame2 = data_frame         
    return data_frame2

In [5]:
#Clean the date columns for the clarity data
clarity_clean_v1 = clarity_col_renamed.copy()
clarity_clean_v1 = date_converter(clarity_clean_v1)

#Clean the quantaq date columns 
quantaq_clean_v1 = quantaq_col_renamed.copy()
quantaq_clean_v1  = date_converter(quantaq_clean_v1 )


In [6]:
#Test to see wether the dates columsn are matching 
#First filter the clarity data to have only one device reuslts 
clarity_clean_49L = clarity_clean_v1[clarity_clean_v1['source_id_clarity']=='A93NM49L'].reset_index(drop=True)
combined_data = pd.concat([clarity_clean_49L,quantaq_clean_v1], axis=1)
combined_data_dates = combined_data[[col for col in combined_data.columns if 'period' in col]]
combined_data_dates

#Compare the start time columns 
combined_data_dates.start_of_period_utc_clarity.equals(combined_data_dates.start_of_period_utc_qa)

#Compare the end time columns
combined_data_dates.end_of_period_utc_clarity.equals(combined_data_dates.end_of_period_utc_qa)

True

In [93]:
#Save the cleaned versions to be analysed in tableau
clarity_clean_v1.to_csv("data-viz/clarity-clean-11-15-25.csv")
quantaq_clean_v1 .to_csv("data-viz/quantaq-clean-11-15-25.csv")

In [57]:
col1 = quantaq_clean_v1.start_of_period_utc_qa
col2 = clarity_clean_v1.start_of_period_utc_clarity[clarity_clean_v1['source_id_clarity']=="A93NM49L"].reset_index()
pd.DataFrame(pd.concat([col1,col2], axis=1, ignore_index=True))

,0,1,2
0,2025-11-09 05:00:00+00:00,0,2025-11-09 05:00:00+00:00
1,2025-11-09 06:00:00+00:00,2,2025-11-09 06:00:00+00:00
2,2025-11-09 07:00:00+00:00,4,2025-11-09 07:00:00+00:00
3,2025-11-09 08:00:00+00:00,6,2025-11-09 08:00:00+00:00
4,2025-11-09 09:00:00+00:00,8,2025-11-09 09:00:00+00:00
5,2025-11-09 10:00:00+00:00,10,2025-11-09 10:00:00+00:00
6,2025-11-09 11:00:00+00:00,12,2025-11-09 11:00:00+00:00
7,2025-11-09 12:00:00+00:00,14,2025-11-09 12:00:00+00:00
8,2025-11-09 13:00:00+00:00,16,2025-11-09 13:00:00+00:00
9,2025-11-09 14:00:00+00:00,18,2025-11-09 14:00:00+00:00
